In [1]:
import os
os.chdir("../../web_backend/")

In [2]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, GraphSearcher, TextEmbeddingSearcher, EmbeddingSearcher, Equaliser

In [3]:
from app import init_DMG, search_collection

In [4]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images/"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")
    print(time_stamp)

    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

    kg_searcher = GraphSearcher(df)


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=None)
    concept_search = TextEmbeddingSearcher(sem_embs, name="concept-searcher")


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=32)
    sem_searcher = EmbeddingSearcher(sem_embs, name="semantic-searcher")
    
    viz_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/vitmae", loadXD=32)
    viz_searcher = EmbeddingSearcher(viz_embs, name="visual-searcher")

    s = Search([kg_searcher, sem_searcher, viz_searcher])
    return df, s, concept_search

df, s, cs = init_DMG()

2026-03-28


[GraphSearcher]: building graph...: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21030/21030 [00:02<00:00, 10337.98it/s]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [5]:
r = df.sample(4)

In [6]:
scores = s(r)

In [7]:
ordered = s.order(df, scores=scores)

In [8]:
Equaliser(df)(r).sort_values() # USED TO BE: Randomiser(cur_coll)

object_number
1992-0004_0-2    0.000048
1992-0004_1-2    0.000048
1992-0004_2-2    0.000048
5051             0.000048
5052             0.000048
                   ...   
2024-0004_0-4    0.000048
DMG_T_01424      0.000048
2022-0018_2-3    0.000048
2022-0018_3-3    0.000048
2022-0054_2-2    0.000048
Name: Equaliser, Length: 21030, dtype: float64

---

In [9]:
ordered.coll.get_presentation_records(as_json=True)

[{'inventory_number': '1986-0035_2-4',
  'title': 'Prototype van een kegelvormige vaas',
  'description': '',
  'designer': 'Pieter Stockmans',
  'producer': 'Mosa',
  'design_date': '1980 — 1980',
  'production_date': '',
  'design_place': 'Genk',
  'production_place': 'Maastricht',
  'rights_attribution': 'In Copyright',
  'image': None,
  'order_index': 0},
 {'inventory_number': '5030',
  'title': 'Jabot van een machinale imitatie van Chantilly kant, bijeengehouden door een lint. Eind 19de, begin 20ste eeuw.',
  'description': 'Op een centraal verlopende zwart (kunst?)zijden lint zijn twee stroken zwarte machinale imitatie Chantillykant gefronseld gemonteerd tot een breed V-vormig jabot.',
  'designer': 'onbekend',
  'producer': 'onbekend',
  'design_date': '',
  'production_date': 'ca. 1930 — ca. 1949',
  'design_place': 'onbekend',
  'production_place': 'België',
  'rights_attribution': 'In Copyright',
  'image': None,
  'order_index': 1},
 {'inventory_number': '2899',
  'title': 

In [ ]:
DMG_DIR = "./data/DMG"
image_folder = DMG_DIR+"/images/"

image_info = pd.read_csv(image_folder+"image_info.csv").set_index("object_number")
